In [2]:
# ============================================================
# INDIAN QUANT PORTFOLIO & RISK ENGINE
#
# STEP 5:
# CLASSICAL vs MACHINE LEARNING PORTFOLIO COMPARISON
#
# Classical Strategies:
#   1. Equal Weight
#   2. Minimum Variance
#   3. Maximum Sharpe
#   4. Risk Parity
#
# ML Strategies:
#   5. Random Forest
#   6. XGBoost
#
# Prediction Target:
#   Forward 21-trading-day return
#
# Backtest:
#   Walk-forward
#   504-day training window
#   21-day rebalancing
#   Transaction costs
#   Long-only
#   Maximum 15% per stock
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import warnings

import numpy as np
import pandas as pd

from scipy.optimize import minimize

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")


# ============================================================
# 2. CONFIGURATION
# ============================================================

DATA_DIR = "data/raw"

OUTPUT_DIR = (
    "data/processed/final_backtest"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

TRADING_DAYS = 252

# Predict approximately one month ahead
HORIZON = 21

# Approximately two years of training data
TRAINING_WINDOW = 504

# Rebalance every month
REBALANCE_DAYS = 21

# Estimated transaction cost
TRANSACTION_COST = 0.001

# Portfolio constraints
MIN_WEIGHT = 0.00
MAX_WEIGHT = 0.15

# Initial risk-free assumption
RISK_FREE_RATE = 0.00

RANDOM_STATE = 42


# ============================================================
# 3. LOAD DATA
# ============================================================

print("=" * 75)
print("LOADING DATA")
print("=" * 75)

prices = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "adjusted_close_prices.csv"
    ),
    index_col="Date",
    parse_dates=True
)

returns = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "daily_returns.csv"
    ),
    index_col="Date",
    parse_dates=True
)


# Sort
prices = prices.sort_index()
returns = returns.sort_index()


# Remove infinite values
prices = prices.replace(
    [np.inf, -np.inf],
    np.nan
)

returns = returns.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 4. SELECT COMMON INVESTMENT UNIVERSE
# ============================================================

minimum_observations = int(
    len(prices) * 0.90
)

stocks = [
    stock
    for stock in prices.columns
    if prices[stock].count()
    >= minimum_observations
]

prices = prices[stocks]

returns = returns[stocks]


print(
    f"\nNumber of stocks: "
    f"{len(stocks)}"
)

print(
    f"Date range: "
    f"{prices.index.min().date()} "
    f"→ "
    f"{prices.index.max().date()}"
)


# ============================================================
# 5. FEATURE ENGINEERING
# ============================================================

def create_features(
    price,
    daily_return
):

    df = pd.DataFrame(
        index=price.index
    )

    # --------------------------------------------------------
    # Momentum
    # --------------------------------------------------------

    df["return_5d"] = (
        price.pct_change(5)
    )

    df["return_21d"] = (
        price.pct_change(21)
    )

    df["return_63d"] = (
        price.pct_change(63)
    )

    df["return_126d"] = (
        price.pct_change(126)
    )

    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------

    df["volatility_21d"] = (
        daily_return
        .rolling(21)
        .std()
        * np.sqrt(TRADING_DAYS)
    )

    df["volatility_63d"] = (
        daily_return
        .rolling(63)
        .std()
        * np.sqrt(TRADING_DAYS)
    )

    # --------------------------------------------------------
    # Moving-average relationships
    # --------------------------------------------------------

    ma_21 = (
        price
        .rolling(21)
        .mean()
    )

    ma_63 = (
        price
        .rolling(63)
        .mean()
    )

    ma_126 = (
        price
        .rolling(126)
        .mean()
    )

    df["price_vs_ma21"] = (
        price / ma_21 - 1
    )

    df["price_vs_ma63"] = (
        price / ma_63 - 1
    )

    df["price_vs_ma126"] = (
        price / ma_126 - 1
    )

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    delta = price.diff()

    gains = delta.clip(
        lower=0
    )

    losses = -delta.clip(
        upper=0
    )

    avg_gain = (
        gains
        .rolling(14)
        .mean()
    )

    avg_loss = (
        losses
        .rolling(14)
        .mean()
    )

    rs = (
        avg_gain
        / avg_loss.replace(
            0,
            np.nan
        )
    )

    df["RSI_14"] = (
        100
        - (
            100
            / (1 + rs)
        )
    )

    # --------------------------------------------------------
    # Drawdown
    # --------------------------------------------------------

    rolling_high = (
        price
        .rolling(126)
        .max()
    )

    df["drawdown_126d"] = (
        price
        / rolling_high
        - 1
    )

    # --------------------------------------------------------
    # Distribution characteristics
    # --------------------------------------------------------

    df["skewness_63d"] = (
        daily_return
        .rolling(63)
        .skew()
    )

    df["kurtosis_63d"] = (
        daily_return
        .rolling(63)
        .kurt()
    )

    return df


# ============================================================
# 6. CREATE FORWARD RETURN TARGET
# ============================================================

def create_target(price):

    return (
        price.shift(-HORIZON)
        / price
        - 1
    )


FEATURES = [

    "return_5d",

    "return_21d",

    "return_63d",

    "return_126d",

    "volatility_21d",

    "volatility_63d",

    "price_vs_ma21",

    "price_vs_ma63",

    "price_vs_ma126",

    "RSI_14",

    "drawdown_126d",

    "skewness_63d",

    "kurtosis_63d"
]


# ============================================================
# 7. BUILD FEATURE DATA
# ============================================================

print("\nCreating features...")

feature_data = {}

target_data = {}

panel_records = []


for stock in stocks:

    feature_df = create_features(
        prices[stock],
        returns[stock]
    )

    target = create_target(
        prices[stock]
    )

    feature_data[
        stock
    ] = feature_df

    target_data[
        stock
    ] = target

    temp = feature_df.copy()

    temp["target"] = target

    temp["stock"] = stock

    temp = temp.dropna()

    panel_records.append(
        temp
    )


ml_data = pd.concat(
    panel_records
).sort_index()


print(
    f"ML observations: "
    f"{len(ml_data):,}"
)


# ============================================================
# 8. MACHINE LEARNING MODELS
# ============================================================

def create_random_forest():

    return RandomForestRegressor(

        n_estimators=300,

        max_depth=6,

        min_samples_leaf=20,

        max_features="sqrt",

        random_state=RANDOM_STATE,

        n_jobs=-1
    )


def create_xgboost():

    return XGBRegressor(

        n_estimators=300,

        max_depth=3,

        learning_rate=0.03,

        subsample=0.8,

        colsample_bytree=0.8,

        min_child_weight=10,

        reg_alpha=0.1,

        reg_lambda=1.0,

        objective="reg:squarederror",

        random_state=RANDOM_STATE,

        n_jobs=-1
    )


ML_MODELS = {

    "Random_Forest":
        create_random_forest,

    "XGBoost":
        create_xgboost
}


# ============================================================
# 9. PORTFOLIO MATH
# ============================================================

def portfolio_volatility(
    weights,
    covariance
):

    variance = (
        weights.T
        @ covariance
        @ weights
    )

    return np.sqrt(
        max(
            variance,
            1e-12
        )
    )


def portfolio_sharpe(
    weights,
    expected_returns,
    covariance
):

    volatility = (
        portfolio_volatility(
            weights,
            covariance
        )
    )

    return (
        (
            weights
            @ expected_returns
        )
        / volatility
    )


def risk_contribution(
    weights,
    covariance
):

    volatility = (
        portfolio_volatility(
            weights,
            covariance
        )
    )

    marginal = (
        covariance
        @ weights
    )

    contribution = (
        weights
        * marginal
    )

    return (
        contribution
        / volatility
    )


# ============================================================
# 10. CLASSICAL PORTFOLIO OPTIMIZERS
# ============================================================

def equal_weight(
    expected_returns,
    covariance
):

    n = len(
        expected_returns
    )

    return (
        np.ones(n)
        / n
    )


def minimum_variance(
    expected_returns,
    covariance
):

    n = len(
        expected_returns
    )

    initial = (
        np.ones(n)
        / n
    )

    bounds = [
        (
            MIN_WEIGHT,
            MAX_WEIGHT
        )
        for _ in range(n)
    ]

    constraint = {
        "type": "eq",
        "fun": lambda w:
            np.sum(w) - 1
    }

    result = minimize(

        lambda w:
        w.T
        @ covariance
        @ w,

        initial,

        method="SLSQP",

        bounds=bounds,

        constraints=constraint,

        options={
            "maxiter": 2000,
            "ftol": 1e-9
        }
    )

    if not result.success:

        return initial

    return result.x


def maximum_sharpe(
    expected_returns,
    covariance
):

    n = len(
        expected_returns
    )

    initial = (
        np.ones(n)
        / n
    )

    bounds = [
        (
            MIN_WEIGHT,
            MAX_WEIGHT
        )
        for _ in range(n)
    ]

    constraint = {
        "type": "eq",
        "fun": lambda w:
            np.sum(w) - 1
    }

    result = minimize(

        lambda w:
        -portfolio_sharpe(
            w,
            expected_returns,
            covariance
        ),

        initial,

        method="SLSQP",

        bounds=bounds,

        constraints=constraint,

        options={
            "maxiter": 2000,
            "ftol": 1e-9
        }
    )

    if not result.success:

        return initial

    return result.x


def risk_parity(
    expected_returns,
    covariance
):

    n = len(
        expected_returns
    )

    initial = (
        np.ones(n)
        / n
    )

    bounds = [
        (
            MIN_WEIGHT,
            MAX_WEIGHT
        )
        for _ in range(n)
    ]

    constraint = {
        "type": "eq",
        "fun": lambda w:
            np.sum(w) - 1
    }

    def objective(weights):

        rc = risk_contribution(
            weights,
            covariance
        )

        target = 1 / n

        return np.sum(
            (
                rc - target
            ) ** 2
        )

    result = minimize(

        objective,

        initial,

        method="SLSQP",

        bounds=bounds,

        constraints=constraint,

        options={
            "maxiter": 2000,
            "ftol": 1e-10
        }
    )

    if not result.success:

        return initial

    return result.x


CLASSICAL_MODELS = {

    "Equal_Weight":
        equal_weight,

    "Minimum_Variance":
        minimum_variance,

    "Maximum_Sharpe":
        maximum_sharpe,

    "Risk_Parity":
        risk_parity
}


# ============================================================
# 11. ML PORTFOLIO OPTIMIZER
# ============================================================

def optimize_ml_portfolio(
    predicted_returns,
    covariance
):

    n = len(
        predicted_returns
    )

    initial = (
        np.ones(n)
        / n
    )

    bounds = [
        (
            MIN_WEIGHT,
            MAX_WEIGHT
        )
        for _ in range(n)
    ]

    constraint = {
        "type": "eq",
        "fun": lambda w:
            np.sum(w) - 1
    }

    def objective(weights):

        expected_return = (
            weights
            @ predicted_returns
        )

        volatility = (
            portfolio_volatility(
                weights,
                covariance
            )
        )

        return -(
            expected_return
            / volatility
        )

    result = minimize(

        objective,

        initial,

        method="SLSQP",

        bounds=bounds,

        constraints=constraint,

        options={
            "maxiter": 2000,
            "ftol": 1e-9
        }
    )

    if not result.success:

        return initial

    weights = result.x

    weights[
        np.abs(weights) < 1e-8
    ] = 0

    return (
        weights
        / weights.sum()
    )


# ============================================================
# 12. PERFORMANCE CALCULATION
# ============================================================

def calculate_performance(
    series
):

    series = series.dropna()

    if len(series) == 0:

        return {}

    total_return = (
        1 + series
    ).prod() - 1

    years = (
        len(series)
        / TRADING_DAYS
    )

    cagr = (
        (
            1
            + total_return
        )
        ** (1 / years)
        - 1
    )

    volatility = (
        series.std()
        * np.sqrt(
            TRADING_DAYS
        )
    )

    sharpe = np.nan

    if volatility > 0:

        sharpe = (
            series.mean()
            * TRADING_DAYS
            / volatility
        )

    downside = (
        series
        .clip(upper=0)
        .std()
        * np.sqrt(
            TRADING_DAYS
        )
    )

    sortino = np.nan

    if downside > 0:

        sortino = (
            series.mean()
            * TRADING_DAYS
            / downside
        )

    cumulative = (
        1 + series
    ).cumprod()

    drawdown = (
        cumulative
        / cumulative.cummax()
    ) - 1

    maximum_drawdown = (
        drawdown.min()
    )

    calmar = np.nan

    if maximum_drawdown < 0:

        calmar = (
            cagr
            / abs(
                maximum_drawdown
            )
        )

    var_95 = (
        series.quantile(
            0.05
        )
    )

    cvar_95 = (
        series[
            series <= var_95
        ].mean()
    )

    return {

        "CAGR":
            cagr,

        "Annualized_Volatility":
            volatility,

        "Sharpe_Ratio":
            sharpe,

        "Sortino_Ratio":
            sortino,

        "Maximum_Drawdown":
            maximum_drawdown,

        "Calmar_Ratio":
            calmar,

        "VaR_95":
            var_95,

        "CVaR_95":
            cvar_95,

        "Positive_Days_%":
            (
                series > 0
            ).mean(),

        "Observations":
            len(series)
    }


# ============================================================
# 13. BACKTEST DATES
# ============================================================

dates = prices.index

start_position = max(
    TRAINING_WINDOW,
    126 + HORIZON
)

rebalance_positions = list(
    range(
        start_position,
        len(dates)
        - HORIZON,
        REBALANCE_DAYS
    )
)

print(
    f"\nNumber of rebalancing periods: "
    f"{len(rebalance_positions)}"
)


# ============================================================
# 14. STORAGE
# ============================================================

all_returns = {}

all_weights = {}

all_predictions = {}

model_errors = {}


# ============================================================
# 15. CLASSICAL BACKTEST
# ============================================================

print("\n" + "=" * 75)
print("RUNNING CLASSICAL STRATEGIES")
print("=" * 75)


for strategy_name, optimizer in (
    CLASSICAL_MODELS.items()
):

    print(
        f"\n{strategy_name}"
    )

    strategy_returns = []

    strategy_weights = []

    previous_weights = None

    for position in (
        rebalance_positions
    ):

        current_date = (
            dates[position]
        )

        # ----------------------------------------------------
        # TRAINING WINDOW
        # ----------------------------------------------------

        train = returns.iloc[
            position
            - TRAINING_WINDOW:
            position
        ]

        train = train.dropna()

        if len(train) < 300:

            continue

        # ----------------------------------------------------
        # EXPECTED RETURNS
        # ----------------------------------------------------

        expected_returns = (
            train.mean()
            * TRADING_DAYS
        ).values

        # ----------------------------------------------------
        # COVARIANCE
        # ----------------------------------------------------

        covariance = (
            train.cov()
            .values
            * TRADING_DAYS
        )

        # ----------------------------------------------------
        # OPTIMIZE
        # ----------------------------------------------------

        weights = optimizer(
            expected_returns,
            covariance
        )

        weights = (
            weights
            / weights.sum()
        )

        # ----------------------------------------------------
        # TURNOVER
        # ----------------------------------------------------

        if previous_weights is None:

            turnover = 1.0

        else:

            turnover = np.sum(
                np.abs(
                    weights
                    - previous_weights
                )
            )

        transaction_cost = (
            turnover
            * TRANSACTION_COST
        )

        # ----------------------------------------------------
        # TEST WINDOW
        # ----------------------------------------------------

        test_dates = dates[
            position:
            min(
                position
                + HORIZON,
                len(dates)
            )
        ]

        test_returns = (
            returns
            .loc[
                test_dates,
                stocks
            ]
            .fillna(0)
        )

        portfolio_returns = (
            test_returns
            @ weights
        )

        if len(
            portfolio_returns
        ) > 0:

            portfolio_returns.iloc[
                0
            ] -= transaction_cost

        for date, ret in (
            portfolio_returns.items()
        ):

            strategy_returns.append({

                "Date":
                    date,

                "Return":
                    ret
            })

        # ----------------------------------------------------
        # SAVE WEIGHTS
        # ----------------------------------------------------

        weight_record = (
            dict(
                zip(
                    stocks,
                    weights
                )
            )
        )

        weight_record[
            "Date"
        ] = current_date

        strategy_weights.append(
            weight_record
        )

        previous_weights = (
            weights
        )

    result = pd.DataFrame(
        strategy_returns
    )

    result = (
        result
        .drop_duplicates(
            "Date"
        )
        .set_index(
            "Date"
        )
        .sort_index()
    )

    all_returns[
        strategy_name
    ] = result["Return"]

    all_weights[
        strategy_name
    ] = pd.DataFrame(
        strategy_weights
    )


# ============================================================
# 16. MACHINE LEARNING BACKTEST
# ============================================================

print("\n" + "=" * 75)
print("RUNNING MACHINE LEARNING STRATEGIES")
print("=" * 75)


for model_name, model_factory in (
    ML_MODELS.items()
):

    print(
        f"\n{model_name}"
    )

    strategy_returns = []

    strategy_weights = []

    strategy_predictions = []

    strategy_errors = []

    previous_weights = None

    for iteration, position in enumerate(
        rebalance_positions
    ):

        current_date = (
            dates[position]
        )

        print(
            f"  "
            f"{current_date.date()} "
            f"({iteration + 1}/"
            f"{len(rebalance_positions)})",
            end="\r"
        )

        # ----------------------------------------------------
        # TRAINING WINDOW
        # ----------------------------------------------------

        train_start = dates[
            position
            - TRAINING_WINDOW
        ]

        train_end = current_date

        train_records = ml_data[
            (
                ml_data.index
                >= train_start
            )
            &
            (
                ml_data.index
                < train_end
            )
        ]

        if len(
            train_records
        ) < 300:

            continue

        X_train = (
            train_records[
                FEATURES
            ]
        )

        y_train = (
            train_records[
                "target"
            ]
        )

        # ----------------------------------------------------
        # TRAIN MODEL
        # ----------------------------------------------------

        model = model_factory()

        model.fit(
            X_train,
            y_train
        )

        # ----------------------------------------------------
        # CURRENT FEATURES
        # ----------------------------------------------------

        current_rows = []

        available_stocks = []

        for stock in stocks:

            try:

                row = (
                    feature_data[
                        stock
                    ]
                    .loc[
                        current_date
                    ]
                )

                if (
                    row[
                        FEATURES
                    ]
                    .notna()
                    .all()
                ):

                    current_rows.append(
                        row[
                            FEATURES
                        ].values
                    )

                    available_stocks.append(
                        stock
                    )

            except KeyError:

                continue

        if len(
            available_stocks
        ) < 10:

            continue

        X_current = pd.DataFrame(
            current_rows,
            columns=FEATURES
        )

        # ----------------------------------------------------
        # PREDICT FORWARD RETURNS
        # ----------------------------------------------------

        predictions = (
            model.predict(
                X_current
            )
        )

        predicted_returns = pd.Series(
            predictions,
            index=available_stocks
        )

        # ----------------------------------------------------
        # COVARIANCE
        # ----------------------------------------------------

        train_returns = (
            returns
            .loc[
                train_start:
                train_end,
                available_stocks
            ]
            .dropna()
        )

        if len(
            train_returns
        ) < 100:

            continue

        covariance = (
            train_returns
            .cov()
            .values
            * TRADING_DAYS
        )

        # ----------------------------------------------------
        # ML OPTIMIZATION
        # ----------------------------------------------------

        weights = (
            optimize_ml_portfolio(
                predicted_returns.values,
                covariance
            )
        )

        weights_series = pd.Series(
            weights,
            index=available_stocks
        )

        # ----------------------------------------------------
        # TURNOVER
        # ----------------------------------------------------

        if previous_weights is None:

            turnover = 1.0

        else:

            previous_aligned = (
                previous_weights
                .reindex(
                    available_stocks
                )
                .fillna(0)
            )

            turnover = np.sum(
                np.abs(
                    weights_series
                    - previous_aligned
                )
            )

        transaction_cost = (
            turnover
            * TRANSACTION_COST
        )

        # ----------------------------------------------------
        # TEST WINDOW
        # ----------------------------------------------------

        test_dates = dates[
            position:
            min(
                position
                + HORIZON,
                len(dates)
            )
        ]

        test_returns = (
            returns
            .loc[
                test_dates,
                available_stocks
            ]
            .fillna(0)
        )

        portfolio_returns = (
            test_returns
            .dot(
                weights_series
            )
        )

        if len(
            portfolio_returns
        ) > 0:

            portfolio_returns.iloc[
                0
            ] -= transaction_cost

        for date, ret in (
            portfolio_returns.items()
        ):

            strategy_returns.append({

                "Date":
                    date,

                "Return":
                    ret
            })

        # ----------------------------------------------------
        # SAVE WEIGHTS
        # ----------------------------------------------------

        weight_record = (
            weights_series
            .to_dict()
        )

        weight_record[
            "Date"
        ] = current_date

        strategy_weights.append(
            weight_record
        )

        # ----------------------------------------------------
        # SAVE PREDICTIONS
        # ----------------------------------------------------

        for stock, prediction in (
            predicted_returns.items()
        ):

            strategy_predictions.append({

                "Date":
                    current_date,

                "Stock":
                    stock,

                "Predicted_Return":
                    prediction
            })

        # ----------------------------------------------------
        # MODEL DIAGNOSTICS
        # ----------------------------------------------------

        train_prediction = (
            model.predict(
                X_train
            )
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_train,
                train_prediction
            )
        )

        mae = (
            mean_absolute_error(
                y_train,
                train_prediction
            )
        )

        strategy_errors.append({

            "Date":
                current_date,

            "RMSE":
                rmse,

            "MAE":
                mae,

            "Number_of_Stocks":
                len(
                    available_stocks
                ),

            "Turnover":
                turnover
        })

        previous_weights = (
            weights_series
        )

    # --------------------------------------------------------
    # STORE MODEL RESULTS
    # --------------------------------------------------------

    result = pd.DataFrame(
        strategy_returns
    )

    result = (
        result
        .drop_duplicates(
            "Date"
        )
        .set_index(
            "Date"
        )
        .sort_index()
    )

    all_returns[
        model_name
    ] = result["Return"]

    all_weights[
        model_name
    ] = pd.DataFrame(
        strategy_weights
    )

    all_predictions[
        model_name
    ] = pd.DataFrame(
        strategy_predictions
    )

    model_errors[
        model_name
    ] = pd.DataFrame(
        strategy_errors
    )


# ============================================================
# 17. ALIGN ALL STRATEGIES
# ============================================================

print("\n\n" + "=" * 75)
print("ALIGNING OUT-OF-SAMPLE PERIOD")
print("=" * 75)


all_returns_df = pd.DataFrame(
    all_returns
)

# IMPORTANT:
# Every strategy must be evaluated on
# exactly the same dates.

all_returns_df = (
    all_returns_df
    .dropna(
        how="any"
    )
    .sort_index()
)


print(
    f"\nCommon test period:"
)

print(
    f"{all_returns_df.index.min().date()}"
    f" → "
    f"{all_returns_df.index.max().date()}"
)

print(
    f"Test observations: "
    f"{len(all_returns_df)}"
)


# ============================================================
# 18. FINAL PERFORMANCE COMPARISON
# ============================================================

performance = {}

for strategy in (
    all_returns_df.columns
):

    performance[
        strategy
    ] = calculate_performance(
        all_returns_df[
            strategy
        ]
    )


performance_df = pd.DataFrame(
    performance
).T


# ============================================================
# 19. CUMULATIVE RETURNS
# ============================================================

cumulative_returns = (
    1 + all_returns_df
).cumprod()


# ============================================================
# 20. ML MODEL DIAGNOSTICS
# ============================================================

prediction_summary = {}

for model_name, metrics in (
    model_errors.items()
):

    if len(metrics) == 0:

        continue

    prediction_summary[
        model_name
    ] = {

        "Average_RMSE":
            metrics[
                "RMSE"
            ].mean(),

        "Average_MAE":
            metrics[
                "MAE"
            ].mean(),

        "Average_Turnover":
            metrics[
                "Turnover"
            ].mean(),

        "Average_Stocks":
            metrics[
                "Number_of_Stocks"
            ].mean(),

        "Rebalances":
            len(metrics)
    }


prediction_summary_df = (
    pd.DataFrame(
        prediction_summary
    ).T
)


# ============================================================
# 21. RANKING
# ============================================================

performance_df[
    "Sharpe_Rank"
] = (
    performance_df[
        "Sharpe_Ratio"
    ]
    .rank(
        ascending=False
    )
)

performance_df[
    "CAGR_Rank"
] = (
    performance_df[
        "CAGR"
    ]
    .rank(
        ascending=False
    )
)


# ============================================================
# 22. SAVE FINAL OUTPUTS
# ============================================================

all_returns_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "all_strategy_returns.csv"
    )
)

cumulative_returns.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "all_strategy_cumulative_returns.csv"
    )
)

performance_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_strategy_comparison.csv"
    )
)

prediction_summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ml_model_comparison.csv"
    )
)


# ============================================================
# 23. SAVE WEIGHTS
# ============================================================

for strategy, weights_df in (
    all_weights.items()
):

    weights_df.to_csv(

        os.path.join(
            OUTPUT_DIR,
            f"{strategy}_weights.csv"
        ),

        index=False
    )


# ============================================================
# 24. SAVE ML PREDICTIONS
# ============================================================

for model_name, prediction_df in (
    all_predictions.items()
):

    prediction_df.to_csv(

        os.path.join(
            OUTPUT_DIR,
            f"{model_name}_predictions.csv"
        ),

        index=False
    )


# ============================================================
# 25. SAVE MODEL DIAGNOSTICS
# ============================================================

for model_name, metrics_df in (
    model_errors.items()
):

    metrics_df.to_csv(

        os.path.join(
            OUTPUT_DIR,
            f"{model_name}_diagnostics.csv"
        ),

        index=False
    )


# ============================================================
# 26. PRINT FINAL RESULTS
# ============================================================

print("\n" + "=" * 75)
print("FINAL STRATEGY COMPARISON")
print("=" * 75)


display_columns = [

    "CAGR",

    "Annualized_Volatility",

    "Sharpe_Ratio",

    "Sortino_Ratio",

    "Maximum_Drawdown",

    "Calmar_Ratio",

    "VaR_95",

    "CVaR_95"
]


print(
    performance_df[
        display_columns
    ]
    .sort_values(
        "Sharpe_Ratio",
        ascending=False
    )
    .round(4)
)


print("\n" + "=" * 75)
print("ML MODEL DIAGNOSTICS")
print("=" * 75)


if not prediction_summary_df.empty:

    print(
        prediction_summary_df
        .round(4)
    )


print("\n" + "=" * 75)
print("FINAL PORTFOLIO VALUES")
print("=" * 75)


print(
    cumulative_returns
    .iloc[-1]
    .sort_values(
        ascending=False
    )
    .round(4)
)


# ============================================================
# 27. WINNER
# ============================================================

best_strategy = (
    performance_df[
        "Sharpe_Ratio"
    ]
    .idxmax()
)

best_sharpe = (
    performance_df.loc[
        best_strategy,
        "Sharpe_Ratio"
    ]
)

best_cagr = (
    performance_df.loc[
        best_strategy,
        "CAGR"
    ]
)

best_drawdown = (
    performance_df.loc[
        best_strategy,
        "Maximum_Drawdown"
    ]
)


print("\n" + "=" * 75)
print("BEST STRATEGY")
print("=" * 75)

print(
    f"\nStrategy: "
    f"{best_strategy}"
)

print(
    f"Sharpe: "
    f"{best_sharpe:.3f}"
)

print(
    f"CAGR: "
    f"{best_cagr:.2%}"
)

print(
    f"Maximum Drawdown: "
    f"{best_drawdown:.2%}"
)


print("\n" + "=" * 75)
print("COMPLETE")
print("=" * 75)

print(
    f"\nResults saved to:"
)

print(
    OUTPUT_DIR
)

LOADING DATA

Number of stocks: 15
Date range: 2015-01-01 → 2026-08-19

Creating features...
ML observations: 40,920

Number of rebalancing periods: 112

RUNNING CLASSICAL STRATEGIES

Equal_Weight

Minimum_Variance

Maximum_Sharpe

Risk_Parity

RUNNING MACHINE LEARNING STRATEGIES

Random_Forest
  2026-06-25 (112/112)
XGBoost
  2026-06-25 (112/112)

ALIGNING OUT-OF-SAMPLE PERIOD

Common test period:
2017-01-18 → 2026-07-23
Test observations: 2352

FINAL STRATEGY COMPARISON
                    CAGR  Annualized_Volatility  Sharpe_Ratio  Sortino_Ratio  \
XGBoost           0.4000                 0.1626        2.1520         3.6701   
Random_Forest     0.3434                 0.1610        1.9155         3.2465   
Equal_Weight      0.1809                 0.1545        1.1543         1.8287   
Minimum_Variance  0.1627                 0.1394        1.1519         1.8615   
Maximum_Sharpe    0.1786                 0.1594        1.1108         1.7892   
Risk_Parity       0.1755                 0.